# Generation Evaluation

Evaluate answer generation under three context strategies by calling the existing generation module.

In [6]:
import os
import sys
import json
import statistics
from tqdm.auto import tqdm
from openai import OpenAI

# Add project root to path
sys.path.append("../..")

from app.rag.generator import generate, build_rag_messages
from app.core.config import get_rag_config

# ---- Configuration ----

# MODEL_NAME = "Gemma-4-31B-IT-NVFP4"
MODEL_NAME = "Qwen2.5-VL-7B-Instruct-NVFP4"
LLM_BASE_URL = "http://localhost:8001/v1"

MAX_WORKERS = 8

QA_FILE = f"./results/qa_pairs.jsonl"
GENERAL_QA_FILE = f"./results/general_qa_pairs.jsonl"
RETRIEVAL_FILE = f"./results/retrieval_results.jsonl"
OUTPUT_FILE = f"./results/{MODEL_NAME}/generation_results.jsonl"


## 1. Setup & Load Data

In [7]:
if not os.path.exists(QA_FILE) or not os.path.exists(RETRIEVAL_FILE):
    raise FileNotFoundError("Required files missing. Run qa_generator.ipynb and retrieval_eval.ipynb first.")

with open(QA_FILE, "r", encoding="utf-8") as f:
    qa_pairs = [json.loads(line) for line in f if line.strip()]
with open(RETRIEVAL_FILE, "r", encoding="utf-8") as f:
    # If retrieval is JSONL:
    retrieval_results_list = [json.loads(line) for line in f if line.strip()]
    # Rebuild map from the list
    retrieval_map = { r["question"]: r for r in retrieval_results_list }
    
print(f"Loaded {len(qa_pairs)} QA pairs.")

rag_config = get_rag_config()
system_prompt = rag_config.get("system_prompt", "You are a helpful assistant.")
client = OpenAI(base_url=LLM_BASE_URL, api_key="not-needed")

Loaded 100 QA pairs.


## 2. Sliding Context Wrapper

In [8]:
def sliding_context_generation(question, chunks):
    """Expand the chunks context by taking previous and next chunks from DB, then generate."""
    if not chunks:
        return ""
    
    from app.db.session import get_session_factory
    from sqlalchemy import and_, or_
    from app.db.models import ChunkMetadata
    
    SessionLocal = get_session_factory()
    db = SessionLocal()
    
    try:
        adj_conditions = []
        for c in chunks:
            if "document_id" in c and "chunk_index" in c:
                adj_conditions.append(
                    and_(
                        ChunkMetadata.document_id == c["document_id"],
                        ChunkMetadata.chunk_index.in_([c["chunk_index"] - 1, c["chunk_index"] + 1])
                    )
                )
        
        adj_map = {}
        if adj_conditions:
            adj_chunks = db.query(ChunkMetadata).filter(or_(*adj_conditions)).all()
            for ac in adj_chunks:
                adj_map[(ac.document_id, ac.chunk_index)] = ac.content
        
        expanded_chunks = []
        for c in chunks:
            chunk = c.copy()
            if "document_id" in chunk and "chunk_index" in chunk:
                doc_id = chunk["document_id"]
                c_idx = chunk["chunk_index"]
                
                prev_content = adj_map.get((doc_id, c_idx - 1))
                next_content = adj_map.get((doc_id, c_idx + 1))
                
                parts = []
                if prev_content:
                    parts.append(prev_content)
                parts.append(chunk.get("content", ""))
                if next_content:
                    parts.append(next_content)
                
                chunk["content"] = "\n\n".join(parts)
            expanded_chunks.append(chunk)
    finally:
        db.close()
    
    messages = build_rag_messages(question, expanded_chunks, system_prompt)
    return generate(messages, max_new_tokens=512, model_path=MODEL_NAME)


## 4. Evaluate Generation Strategies

In [9]:
import concurrent.futures

from app.rag.generator import build_rag_messages_base

def process_generation_query(qa, retrieval_map, model_name, system_prompt):
    """Evaluates all generation strategies for a single question."""
    q = qa["question"]
    gt_ans = qa["answer"]
    
    ret_res = retrieval_map.get(q)
    if not ret_res:
        return None
        
    hybrid_chunks = ret_res["hybrid_rerank"]["retrieved_details"]
    
    # 1. No System Prompt
    messages_no_sys = build_rag_messages_base(q,system_prompt="")
    ans_no_sys = generate(messages_no_sys, max_new_tokens=512, model_path=model_name)

    # 2. Base Knowledge
    messages_base = build_rag_messages_base(q, system_prompt)
    ans_base = generate(messages_base, max_new_tokens=512, model_path=model_name)
    
    # 3. + Context
    messages_ctx = build_rag_messages(q, hybrid_chunks, system_prompt)
    ans_ctx = generate(messages_ctx, max_new_tokens=512, model_path=model_name)
    ctx_len = sum(len(c.get("content", "")) for c in hybrid_chunks)
    
    # 4. + Sliding Context
    ans_slide = sliding_context_generation(q, hybrid_chunks)
    
    return {
        "question": q,
        "ground_truth_answer": gt_ans,
        "strategies": {
            "no_system_prompt": {"answer": ans_no_sys},
            "base_knowledge": {"answer": ans_base},
            "with_context": {"answer": ans_ctx, "context_length": ctx_len},
            "sliding_context": {"answer": ans_slide}
        },
        "retrieved_chunk_ids": [c.get("chunk_id") for c in hybrid_chunks]
    }

# 1. Clear output file
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    pass

# 2. Results container for summary
per_question_results = []

print(f"Starting Threaded Generation Eval with {MAX_WORKERS} workers...")

with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    future_to_qa = {
        executor.submit(process_generation_query, qa, retrieval_map, MODEL_NAME, system_prompt): qa 
        for qa in qa_pairs
    }
    
    for future in tqdm(concurrent.futures.as_completed(future_to_qa), total=len(qa_pairs), desc="Generation Progress"):
        res = future.result()
        if res:
            # Save immediately to JSONL
            with open(OUTPUT_FILE, "a", encoding="utf-8") as f:
                f.write(json.dumps(res, ensure_ascii=False) + "\n")
            
            # Store for the final stats summary
            per_question_results.append(res)

print(f"\nDone! Processed {len(per_question_results)} questions.")

# Update the results object for the final summary cell to use
generation_eval_results = {
    "summary": {
        "model": MODEL_NAME,
        "total_evaluated": len(per_question_results)
    },
    "per_question_results": per_question_results
}


Starting Threaded Generation Eval with 8 workers...


Generation Progress:   0%|          | 0/100 [00:00<?, ?it/s]


Done! Processed 100 questions.


In [10]:
with open("done.txt", "w") as f:
    f.write("done-generation-eval-qwen")